In [22]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from collections import Counter
import warnings, gc
warnings.filterwarnings('ignore')

SEED       = 42
N_FOLDS    = 10
TARGET     = 'Irrigation_Need'
target_map  = {'Low': 0, 'Medium': 1, 'High': 2}
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}
COMP = '/home/chotu/Downloads/Irrigation_Submission/data/raw'
ORIG = '/home/chotu/Downloads/Irrigation_Submission/data/OG_data'

In [15]:
import os
import numpy as np
import pandas as pd
from collections import Counter

# --- Assume COMP, ORIG, TARGET, and target_map are defined above in your notebook ---
# Example:
# COMP = '/kaggle/input/playground-series-s6e4'
# ORIG = '/kaggle/input/irrigation-dataset'
# TARGET = 'Irrigation_Need'
# target_map = {0: 0, 1: 1} # Update with your actual mapping

# 1. Load base datasets using robust path joining
train = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/raw/train.csv')
test  = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/raw/test.csv')
orig  = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/OG_data/irrigation_prediction.csv')

# 2. Setup pseudo-labeling directory
sub_dir = '/home/chotu/Downloads/Irrigation_Submission/pseudolabel/nina_labels' 
sub_files = sorted([f for f in os.listdir(sub_dir) if f.endswith('.csv')])
print(f'Found {len(sub_files)} public submissions for voting')

# 3. Vote across all public submissions
vote_preds = []
for f in sub_files:
    # Use os.path.join to prevent FileNotFoundError
    df = pd.read_csv(os.path.join(sub_dir, f))
    if 'Irrigation_Need' in df.columns:
        vote_preds.append(df['Irrigation_Need'].values)

vote_matrix = np.array(vote_preds)
n_rows = vote_matrix.shape[1]

# 4. Calculate majority vote and confidence ratio
pseudo_labels = []
confidence    = []
for i in range(n_rows):
    c = Counter(vote_matrix[:, i])
    top_label, top_count = c.most_common(1)[0]
    pseudo_labels.append(top_label)
    confidence.append(top_count / len(vote_preds))

test['pseudo_label'] = pseudo_labels
test['confidence']   = confidence

# 5. Filter for high confidence pseudo-labels (>= 90% agreement)
high_conf = test[test['confidence'] >= 0.90].copy()
print(f"High confidence pseudo-labels (>=90% agreement): {len(high_conf):,} / {len(test):,}")
print(f"Pseudo-label distribution: {Counter(high_conf['pseudo_label'])}")

# 6. Apply target mapping
train[TARGET] = train[TARGET].map(target_map)
orig[TARGET]  = orig[TARGET].map(target_map)
high_conf['pseudo_label_enc'] = high_conf['pseudo_label'].map(target_map)

Found 22 public submissions for voting
High confidence pseudo-labels (>=90% agreement): 268,619 / 270,000
Pseudo-label distribution: Counter({'Low': 159245, 'Medium': 99542, 'High': 9832})


In [16]:
CATS = ['Soil_Type','Crop_Type','Crop_Growth_Stage','Season',
        'Irrigation_Type','Water_Source','Mulching_Used','Region']
NUMS = ['Soil_pH','Soil_Moisture','Organic_Carbon','Electrical_Conductivity',
        'Temperature_C','Humidity','Rainfall_mm','Sunlight_Hours',
        'Wind_Speed_kmh','Field_Area_hectare','Previous_Irrigation_mm']

def add_features(df_list):
    for df in df_list:
        eps = 1e-6
        df['water_avail']  = df['Soil_Moisture'] + df['Rainfall_mm'] / 300.0
        df['heat_stress']  = df['Temperature_C'] / (df['Soil_Moisture'] + eps)
        df['rain_cool']    = df['Rainfall_mm'] / (df['Temperature_C'] + eps)
        df['wind_evap']    = df['Wind_Speed_kmh'] * df['Temperature_C'] / 100.0
        df['soil_x_rain']  = df['Soil_Moisture'] * df['Rainfall_mm'] / 1000.0
        df['temp_x_dry']   = df['Temperature_C'] * (100 - df['Soil_Moisture']) / 100.0
        df['soil_lt_25']   = (df['Soil_Moisture'] < 25).astype(int)
        df['temp_gt_30']   = (df['Temperature_C'] > 30).astype(int)
        df['rain_lt_300']  = (df['Rainfall_mm'] < 300).astype(int)
        df['wind_gt_10']   = (df['Wind_Speed_kmh'] > 10).astype(int)
        df['logit_low']    = (16.3173 - 11.0237*df['soil_lt_25'] - 5.8559*df['temp_gt_30']
                              - 10.8500*df['rain_lt_300'] - 5.8284*df['wind_gt_10'])
        df['logit_high']   = (-20.9697 + 10.6947*df['soil_lt_25'] + 5.8763*df['temp_gt_30']
                              + 10.6958*df['rain_lt_300'] + 5.7444*df['wind_gt_10'])
        df['crop_stage']   = df['Crop_Type'].astype(str)+'_'+df['Crop_Growth_Stage'].astype(str)
        df['soil_region']  = df['Soil_Type'].astype(str)+'_'+df['Region'].astype(str)
        df['season_irr']   = df['Season'].astype(str)+'_'+df['Irrigation_Type'].astype(str)
    return df_list

add_features([train, test, orig, high_conf])
CAT_COMBOS = ['crop_stage','soil_region','season_irr']
ALL_CATS   = CATS + CAT_COMBOS
NEW_NUMS   = ['water_avail','heat_stress','rain_cool','wind_evap','soil_x_rain',
              'temp_x_dry','soil_lt_25','temp_gt_30','rain_lt_300','wind_gt_10',
              'logit_low','logit_high']

# Label encode
for c in ALL_CATS:
    le = LabelEncoder()
    combined = pd.concat([train[c], test[c], orig[c]]).astype(str)
    le.fit(combined)
    for df in [train, test, orig, high_conf]:
        df[c] = le.transform(df[c].astype(str))

# Orig-based TE
TE_FEATS = []
for c in CATS:
    for cls in [0, 1, 2]:
        col = f'TE_{c}_cls{cls}'
        te_map = orig.groupby(c)[TARGET].apply(lambda x: (x==cls).mean())
        for df in [train, test, orig, high_conf]:
            df[col] = df[c].map(te_map).fillna(1/3)
        TE_FEATS.append(col)

ALL_FEATS = list(dict.fromkeys(NUMS + NEW_NUMS + ALL_CATS + TE_FEATS))
print(f'Total features: {len(ALL_FEATS)}')

Total features: 58


In [17]:
# Combine: original train + orig dataset + HIGH CONFIDENCE pseudo-labeled test
X_train  = train[ALL_FEATS].values.astype(np.float32)
y_train  = train[TARGET].values
X_orig   = orig[ALL_FEATS].values.astype(np.float32)
y_orig   = orig[TARGET].values
X_pseudo = high_conf[ALL_FEATS].values.astype(np.float32)
y_pseudo = high_conf['pseudo_label_enc'].values
X_te     = test[ALL_FEATS].values.astype(np.float32)

# Pseudo-labels get lower weight (0.6) since they might be wrong
pseudo_weight = 0.6
print(f"Training data sizes:")
print(f"  Original train:  {len(X_train):,}")
print(f"  Orig dataset:    {len(X_orig):,}")
print(f"  Pseudo-labeled:  {len(X_pseudo):,} (weight={pseudo_weight})")
print(f"  Total effective: ~{len(X_train) + len(X_orig) + int(len(X_pseudo)*pseudo_weight):,}")

Training data sizes:
  Original train:  630,000
  Orig dataset:    10,000
  Pseudo-labeled:  268,619 (weight=0.6)
  Total effective: ~801,171


In [18]:
lgb_params = {
    'objective': 'multiclass', 'num_class': 3,
    'metric': 'multi_logloss', 'boosting_type': 'gbdt',
    'n_estimators': 2000, 'learning_rate': 0.05,
    'num_leaves': 63, 'min_child_samples': 20,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'class_weight': 'balanced',
    'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_lgb  = np.zeros((len(X_train), 3))
test_lgb = np.zeros((len(X_te), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'LGB Fold {fold+1}/{N_FOLDS}', end=' ')

    # Build augmented training fold
    Xtr = np.vstack([X_train[tr_idx], X_orig, X_pseudo])
    ytr = np.concatenate([y_train[tr_idx], y_orig, y_pseudo])

    # Sample weights: train=1.0, orig=1.0, pseudo=0.6
    sw = np.concatenate([
        compute_sample_weight('balanced', y_train[tr_idx]),
        compute_sample_weight('balanced', y_orig),
        np.full(len(y_pseudo), pseudo_weight)
    ])

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(Xtr, ytr, sample_weight=sw,
              eval_set=[(X_train[val_idx], y_train[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])

    oof_lgb[val_idx]  = model.predict_proba(X_train[val_idx])
    test_lgb         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y_train[val_idx], oof_lgb[val_idx].argmax(1))
    print(f'| BA={score:.5f}')
    del model; gc.collect()

lgb_score = balanced_accuracy_score(y_train, oof_lgb.argmax(1))
print(f'LGB OOF: {lgb_score:.5f}')

LGB Fold 1/10 | BA=0.97111
LGB Fold 2/10 | BA=0.96847
LGB Fold 3/10 | BA=0.97069
LGB Fold 4/10 | BA=0.97177
LGB Fold 5/10 | BA=0.97076
LGB Fold 6/10 | BA=0.97301
LGB Fold 7/10 | BA=0.97086
LGB Fold 8/10 | BA=0.96919
LGB Fold 9/10 | BA=0.97088
LGB Fold 10/10 | BA=0.96972
LGB OOF: 0.97065


In [19]:
cb_params = dict(
    iterations=1500, learning_rate=0.05, depth=8,
    loss_function='MultiClass', eval_metric='Accuracy',
    class_weights=[1.0, 1.0, 8.0],   # boost High class hard
    random_seed=SEED, verbose=0,
    task_type='GPU', early_stopping_rounds=50,
    border_count=32, bootstrap_type='Bernoulli', subsample=0.8,
)

oof_cat  = np.zeros((len(X_train), 3))
test_cat = np.zeros((len(X_te), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'CAT Fold {fold+1}/{N_FOLDS}', end=' ')
    Xtr = np.vstack([X_train[tr_idx], X_orig, X_pseudo])
    ytr = np.concatenate([y_train[tr_idx], y_orig, y_pseudo])
    sw  = np.concatenate([
        compute_sample_weight('balanced', y_train[tr_idx]),
        compute_sample_weight('balanced', y_orig),
        np.full(len(y_pseudo), pseudo_weight)
    ])

    model = cb.CatBoostClassifier(**cb_params)
    model.fit(Xtr, ytr, sample_weight=sw,
              eval_set=(X_train[val_idx], y_train[val_idx]),
              use_best_model=True)
    oof_cat[val_idx]  = model.predict_proba(X_train[val_idx])
    test_cat         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y_train[val_idx], oof_cat[val_idx].argmax(1))
    print(f'| BA={score:.5f}')
    del model; gc.collect()

cat_score = balanced_accuracy_score(y_train, oof_cat.argmax(1))
print(f'CAT OOF: {cat_score:.5f}')

CAT Fold 1/10 | BA=0.96627
CAT Fold 2/10 | BA=0.96593
CAT Fold 3/10 | BA=0.96653
CAT Fold 4/10 | BA=0.96823
CAT Fold 5/10 | BA=0.96727
CAT Fold 6/10 | BA=0.96956
CAT Fold 7/10 | BA=0.96566
CAT Fold 8/10 | BA=0.96890
CAT Fold 9/10 | BA=0.96715
CAT Fold 10/10 | BA=0.96732
CAT OOF: 0.96728


In [24]:
# Average LGB + CatBoost
final_probs = (test_lgb + test_cat) / 2
oof_avg     = (oof_lgb  + oof_cat)  / 2
avg_score   = balanced_accuracy_score(y_train, oof_avg.argmax(1))
print(f'Ensemble OOF: {avg_score:.5f}')
print(pd.Series([reverse_map[p] for p in final_probs.argmax(1)]).value_counts().to_dict())

sub = pd.read_csv(COMP + '/sample_submission.csv')
sub[TARGET] = [reverse_map[p] for p in final_probs.argmax(1)]
sub.to_csv('submission_pseudolabel.csv', index=False)
print('submission_pseudolabel.csv saved!')

# Soft probabilities for ensembling with V2
proba_df = pd.DataFrame(final_probs, columns=['proba_Low','proba_Medium','proba_High'])
proba_df.insert(0, 'id', sub['id'].values)
proba_df.to_csv('soft_proba_pseudolabel.csv', index=False)
print('soft_proba_pseudolabel.csv saved — bring this back for soft ensemble!')

Ensemble OOF: 0.97159
{'Low': 159890, 'Medium': 99804, 'High': 10306}
submission_pseudolabel.csv saved!
soft_proba_pseudolabel.csv saved — bring this back for soft ensemble!
